<a href="https://colab.research.google.com/github/KModel212/RAG-FAHMAI-SUPERAI/blob/main/hack3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U pythainlp rank_bm25 sentence-transformers FlagEmbedding==1.3.2 transformers==4.44.2 langchain-text-splitters

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.8/177.8 kB 5.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 70.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 118.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 3.3 MB/s eta 0:00:00

In [ ]:
# Load the data from kaggle and upload here.
!unzip super-ai-engineer-s-6-fah-mai-rag-challenge-level-1

Archive:  super-ai-engineer-s-6-fah-mai-rag-challenge-level-1.zip
  inflating: data/knowledge_base/policies/cancellation_policy.md  
  inflating: data/knowledge_base/policies/membership_points_policy.md  
  inflating: data/knowledge_base/policies/return_policy.md  
  inflating: data/knowledge_base/policies/shipping_policy.md  
  inflating: data/knowledge_base/policies/warranty_policy.md  
  inflating: data/knowledge_base/products/AW-MN-001_arcwave_proview_27_4k.md  
  inflating: data/knowledge_base/products/AW-SK-001_arcwave_soundpillar_300.md  
  inflating: data/knowledge_base/products/DN-DT-001_daonuea_tower_x10.md  
  inflating: data/knowledge_base/products/DN-DT-002_daonuea_tower_x10_max.md  
  inflating: data/knowledge_base/products/DN-DT-003_daonuea_mini_pc_m1.md  
  inflating: data/knowledge_base/products/DN-DT-004_daonuea_all_in_one_27.md  
  inflating: data/knowledge_base/products/DN-DT-005_daonuea_all_in_one_24.md  
  inflating: data/knowledge_base/products/DN-LT-001_daonuea_

In [ ]:
import os
import re
import time
import requests
import pandas as pd
import numpy as np
from tqdm import tqdm
from pythainlp.tokenize import word_tokenize
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from google.colab import userdata
from FlagEmbedding import FlagReranker
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. CONFIGURATION & API KEYS

In [ ]:
THAILLM_API_KEY = userdata.get('ThaiLLM')
DATA_PATH = "/content/data/knowledge_base"
QUESTION_FILE = "/content/data/questions.csv"

# 2. CORE FUNCTIONS (ThaiLLM & Parser)

In [ ]:
def ask_llm(messages, model="kbtg", max_retries=5):
    url = f"http://thaillm.or.th/api/{model}/v1/chat/completions"
    headers = {"Content-Type": "application/json", "apikey": THAILLM_API_KEY}
    payload = {
        "model": "/model",
        "messages": messages,
        "max_tokens": 2024,
        "temperature": 0,
    }
    for attempt in range(max_retries):
        try:
            resp = requests.post(url, headers=headers, json=payload, timeout=120)
            if resp.status_code == 429:
                wait = min(2 ** attempt, 30)
                print(f"  Rate limited, waiting {wait}s...")
                time.sleep(wait)
                continue
            resp.raise_for_status()
            return resp.json()["choices"][0]["message"]["content"].strip()
        except Exception as e:
            wait = 2 ** attempt
            print(f"  Error: {e}, retrying in {wait}s...")
            time.sleep(wait)
    return None

def parse_answer(text):
    if text is None: return None
    clean = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    m = re.search(r"ANSWER:\s*(\d+)", clean)
    if m: return int(m.group(1))
    for d in re.findall(r"\b(\d{1,2})\b", clean):
        if 1 <= int(d) <= 10: return int(d)
    return None

# 3. DATA LOADING & PRE-PROCESSING (Chunk 1000)

In [ ]:
documents = []

for root, dirs, files in os.walk(DATA_PATH):
    for file in files:
        if file.endswith(".md") or file.endswith(".txt"):
            with open(os.path.join(root, file), 'r', encoding='utf-8') as f:
                documents.append({"path": file, "text": f.read()})

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", " ", ""]
)

all_chunks = []
for doc in documents:
    chunks_list = text_splitter.split_text(doc['text'])
    for i, content in enumerate(chunks_list):
        all_chunks.append({"id": f"{doc['path']}_{i}", "content": content})

print(f"Created {len(all_chunks)} chunks.")

# 4. INDEXING (BM25 + BGE-M3)

In [ ]:
# BM25 Indexing
tokenized_corpus = [word_tokenize(c['content'], engine="newmm") for c in all_chunks]
bm25 = BM25Okapi(tokenized_corpus)

# BGE-M3 Embedding
embed_model = SentenceTransformer("BAAI/bge-m3")
chunk_texts = [c['content'] for c in all_chunks]
chunk_embeddings = embed_model.encode(chunk_texts, batch_size=16, show_progress_bar=True, normalize_embeddings=True)

# BGE-V2-M3 Reranker
reranker = FlagReranker('BAAI/bge-reranker-v2-m3', use_fp16=True)

# 5. HYBRID SEARCH & RERANK FUNCTION

In [ ]:
def get_context(query, top_k_rerank=5):
    # 1. BM25 Search
    t_query = word_tokenize(query, engine="newmm")
    bm25_scores = bm25.get_scores(t_query)

    # 2. Dense Search
    q_embed = embed_model.encode([query], normalize_embeddings=True)
    dense_scores = np.dot(chunk_embeddings, q_embed.T).flatten()

    # 3. Hybrid (Norm Scores) - ดึงมา 20 เพื่อมา Re-rank
    norm_bm25 = bm25_scores / (np.max(bm25_scores) + 1e-9)
    combined_scores = (0.5 * norm_bm25) + (0.5 * dense_scores)
    top_20_idx = np.argsort(combined_scores)[::-1][:20]
    candidates = [all_chunks[i]['content'] for i in top_20_idx]

    # 4. Reranking (The Decider)
    pairs = [[query, c] for c in candidates]
    rerank_scores = reranker.compute_score(pairs)
    ranked = sorted(zip(candidates, rerank_scores), key=lambda x: x[1], reverse=True)

    return [r[0] for r in ranked[:top_k_rerank]]


# 6. MAIN EXECUTION LOOP

In [ ]:
df_questions = pd.read_csv(QUESTION_FILE)
results = []

for _, row in tqdm(df_questions.iterrows(), total=len(df_questions)):
    qid = row['id']
    query = row['question']

    choices = "\n".join([f"{i}) {row[f'choice_{i}']}" for i in range(1, 11)])

    # Top-k 5
    best_contexts = get_context(query, top_k_rerank=5)
    context_str = "\n---\n".join(best_contexts)

    system_prompt = (
        "คุณคือพนักงานร้าน FahMai ตอบคำถามโดยใช้บริบทที่ให้มาเท่านั้น\n"
        "กฎ: ตอบเพียง 'ANSWER: X' เมื่อ X คือเลขข้อที่เลือก (1-10) เพียงอย่างเดียว"
    )
    user_prompt = f"บริบท:\n{context_str}\n\nคำถาม: {query}\nตัวเลือก:\n{choices}\nตอบ (ANSWER: X):"

    # KBTG Model
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    raw_ans = ask_llm(messages, model="kbtg")
    prediction = parse_answer(raw_ans)

    results.append({"id": qid, "answer": prediction if prediction else 1})

# 7. SAVE SUBMISSION

In [ ]:
submission_df = pd.DataFrame(results)
submission_df.to_csv("submission_scratch_final.csv", index=False)
print("\n🎉 Done! Created 'submission_scratch_final.csv'")

--- Step 1: Loading & Chunking Data ---
Created 514 chunks.
--- Step 2: Indexing (BM25 & BGE-M3) ---


Batches:   0%|          | 0/33 [00:00<?, ?it/s]

--- Step 3: Processing Questions ---


pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 48.27it/s]
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:2888: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 36.80it/s]


  Error: 502 Server Error: Bad Gateway for url: http://thaillm.or.th/api/kbtg/v1/chat/completions, retrying in 1s...


...

  Error: 502 Server Error: Bad Gateway for url: http://thaillm.or.th/api/kbtg/v1/chat/completions, retrying in 1s...


pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 53.00it/s]


  Error: 502 Server Error: Bad Gateway for url: http://thaillm.or.th/api/kbtg/v1/chat/completions, retrying in 1s...


100%|██████████| 100/100 [34:42<00:00, 20.82s/it]


🎉 Done! Created 'submission_scratch_final.csv'


# 8. DOUBLE-CHECK REFINEMENT (Focus on 9 & 10)

In [ ]:
refined_df = submission_df.copy()

risk_ids = refined_df[refined_df['answer'].isin([9, 10])]['id'].tolist()

if not risk_ids:
    print("No high-risk answers found. skipping...")
else:
    for qid in tqdm(risk_ids):
        q_row = df_questions[df_questions['id'] == qid].iloc[0]
        query = q_row['question']
        choices = "\n".join([f"{i}) {q_row[f'choice_{i}']}" for i in range(1, 11)])

        best_contexts = get_context(query, top_k_rerank=10)
        context_str = "\n---\n".join(best_contexts)

        refine_system_prompt = (
            "คุณคือหัวหน้าทีมตรวจสอบข้อมูล (Data Auditor) ของร้าน FahMai\n"
            "หน้าที่: ตรวจสอบคำตอบที่ AI รายงานว่าไม่มีข้อมูล\n"
            "กฎเหล็ก:\n"
            "1. วิเคราะห์บริบทอย่างละเอียดที่สุด แม้จะเป็นข้อมูลทางอ้อมหรือสเปกที่ใกล้เคียง\n"
            "2. หากมีข้อมูลที่พอจะเชื่อมโยงได้ 'ต้อง' เลือกข้อที่ตรงที่สุด ห้ามตอบข้อที่แปลว่าไม่มีข้อมูลเด็ดขาด\n"
            "3. ตอบเพียง 'ANSWER: X' เท่านั้น"
        )

        refine_user_prompt = (
            f"คำถาม: {query}\n"
            f"ตัวเลือก:\n{choices}\n"
            f"บริบทอ้างอิงอย่างละเอียด:\n{context_str}\n\n"
            f"คำแนะนำ: ตรวจสอบชื่อรุ่น ราคา และความแตกต่างของสินค้าให้ดี\n"
            f"ตอบ (ANSWER: X):"
        )

        messages = [
            {"role": "system", "content": refine_system_prompt},
            {"role": "user", "content": refine_user_prompt}
        ]

        raw_ans = ask_llm(messages, model="kbtg")
        new_prediction = parse_answer(raw_ans)

        if new_prediction and new_prediction not in [9, 10]:
            refined_df.loc[refined_df['id'] == qid, 'answer'] = new_prediction

refined_df.to_csv("submission_refined_final.csv", index=False)
print(f"\n🎉 Refinement Complete! Checked {len(risk_ids)} items.")
print("Saved to 'submission_refined_final.csv'")

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 55.75it/s]


  Error: 502 Server Error: Bad Gateway for url: http://thaillm.or.th/api/kbtg/v1/chat/completions, retrying in 1s...


100%|██████████| 12/12 [02:51<00:00, 14.28s/it]


🎉 Refinement Complete! Checked 12 items.
Saved to 'submission_refined_final.csv'
